# Competencia: aves de Colombia, con palabras

**Sesión 8 · IA Intermedio · EstudIA 2026**

Diez aves que se ven en Colombia: azulejo, barranquero, bichofué, canario costeño, colibrí colirrufo, copetón,
gallinazo, mirla, sirirí y torcaza. Tienes **1.000 fotos** de prueba (100 de cada una), tomadas por personas en todo el
país y publicadas en iNaturalist. **No hay fotos etiquetadas.** Tu herramienta principal son las palabras: describes
cada especie y el modelo de imagen y texto hace el resto.

**La métrica: exactitud.** Hay 100 fotos de cada especie, así que la exactitud y la exactitud balanceada dan lo mismo.
Responder siempre la misma especie da 10 %.

**Reglas**

1. Usa un modelo general de imagen y texto, congelado (SigLIP, CLIP…). No valen modelos entrenados para reconocer
   aves, ni entrenar o ajustar un modelo.
2. Puedes etiquetar fotos **tú mismo**, pero solo de `aves_sin_etiqueta.zip` (400 fotos de otras personas), mirándolas.
   Nada de etiquetar las fotos de prueba ni de buscar las fotos en internet.
3. Envía todas las veces que quieras (una vez cada 10 segundos). Gana la mayor **exactitud**.
4. La tabla en vivo usa **la mitad** de las fotos de prueba. Al final, el profesor revela el resultado con **la otra
   mitad**: ese decide la competencia.

**Cómo empezar.** *Archivo › Guardar una copia en Drive*. Con GPU va más rápido (*Entorno de ejecución › Cambiar tipo
de entorno de ejecución › GPU T4*); sin GPU también funciona. Ejecuta todo el notebook (*Entorno de ejecución ›
Ejecutar todas*). Cuando te pida la clave, cópiala de la página de la competencia en la plataforma y pégala.

In [ ]:
# Preparación: librerías y datos. Ejecútala una vez.
import os, re, time, json, zipfile, urllib.request, warnings
warnings.filterwarnings("ignore")
ARCHIVOS = os.environ.get("ESTUDIA_ARCHIVOS", "https://storage.googleapis.com/estud-ia-course-estudia-lab/s08")
MODELO = "google/siglip2-base-patch16-224"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from PIL import Image
torch.set_num_threads(max(1, os.cpu_count() or 1))

def descargar(nombre, url=None):
    """Baja un archivo (de la carpeta pública del curso, o de `url`) solo la primera vez y devuelve su ruta."""
    ruta = os.path.join("archivos_curso", os.path.basename(nombre))
    if not os.path.exists(ruta):
        os.makedirs(os.path.dirname(ruta), exist_ok=True)
        urllib.request.urlretrieve(url or f"{ARCHIVOS}/{nombre}", ruta)
    return ruta

def miles(n):
    """12345 -> "12.345", como escribimos los números en español."""
    return f"{n:,}".replace(",", ".")

def pct(x, decimales=1):
    """0.958 -> "95,8 %"."""
    return f"{100 * x:.{decimales}f} %".replace(".", ",")

def tokens(texto):
    """Parte un texto en palabras y signos, en minúsculas: "No me gustó." -> ["no", "me", "gustó", "."]."""
    return re.findall(r"\w+|[^\w\s]", str(texto).lower())

from transformers import AutoModel, AutoProcessor
from transformers.utils import logging as registro_hf
registro_hf.set_verbosity_error()
DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
_cargados = {}

def _modelo(nombre):
    if nombre not in _cargados:
        _cargados[nombre] = (AutoModel.from_pretrained(nombre).to(DISPOSITIVO).eval(), AutoProcessor.from_pretrained(nombre))
    return _cargados[nombre]

def _unitario(v):
    v = getattr(v, "pooler_output", v)           # según la versión, viene suelto o dentro de un objeto
    return torch.nn.functional.normalize(v, dim=-1).float().cpu().numpy()

def vectores_de_fotos(fotos, modelo=None, lote=32):
    """Un vector de largo 1 por foto (fotos de PIL o rutas a archivos)."""
    red, proc = _modelo(modelo or MODELO)
    salida = []
    with torch.no_grad():
        for i in range(0, len(fotos), lote):
            grupo = [f if isinstance(f, Image.Image) else Image.open(f) for f in fotos[i:i + lote]]
            grupo = [f.convert("RGB") for f in grupo]
            salida.append(_unitario(red.get_image_features(**proc(images=grupo, return_tensors="pt").to(DISPOSITIVO))))
    return np.concatenate(salida)

def vectores_de_textos(textos, modelo=None):
    """Un vector de largo 1 por texto, en el mismo espacio que las fotos."""
    nombre = modelo or MODELO
    red, proc = _modelo(nombre)
    relleno = {"padding": "max_length", "max_length": 64} if "siglip" in nombre else {"padding": True}
    with torch.no_grad():
        entrada = proc(text=list(textos), return_tensors="pt", truncation=True, **relleno).to(DISPOSITIVO)
        return _unitario(red.get_text_features(**entrada))

def mostrar(fotos, titulos=None, columnas=6, tamano=2.0, colores=None):
    """Una cuadrícula de fotos con su título."""
    filas = (len(fotos) + columnas - 1) // columnas
    fig, ejes = plt.subplots(filas, columnas, figsize=(tamano * columnas, tamano * filas + 0.3 * filas))
    for k, eje in enumerate(np.atleast_1d(ejes).ravel()):
        eje.axis("off")
        if k < len(fotos):
            f = fotos[k] if isinstance(fotos[k], Image.Image) else Image.open(fotos[k])
            eje.imshow(f.convert("RGB"))
            if titulos:
                eje.set_title(titulos[k], fontsize=9, color=(colores[k] if colores else "black"))
    plt.tight_layout(); plt.show()

def abrir_zip(nombre):
    """Descarga (la primera vez) y descomprime un zip de la carpeta del curso; devuelve sus fotos ordenadas."""
    ruta = descargar(nombre)
    with zipfile.ZipFile(ruta) as z:
        z.extractall("archivos_curso")
        return sorted(os.path.join("archivos_curso", n) for n in z.namelist() if n.lower().endswith(".jpg"))

from IPython.display import display
REGISTRO = "experimentos.csv"

def registrar(**datos):
    """Agrega una fila a tu registro de experimentos (el archivo experimentos.csv): qué probaste y qué salió."""
    fila = pd.DataFrame([{"hora": time.strftime("%H:%M"), **datos}])
    tabla = pd.concat([pd.read_csv(REGISTRO), fila], ignore_index=True) if os.path.exists(REGISTRO) else fila
    tabla.to_csv(REGISTRO, index=False)

def ver_registro():
    """Muestra el registro de experimentos, el más reciente abajo."""
    if not os.path.exists(REGISTRO):
        print("Todavía no hay experimentos en el registro.")
        return
    tabla = pd.read_csv(REGISTRO)
    display(tabla.style.format(precision=1, decimal=",", thousands=".", na_rep="").hide(axis="index"))

aves = pd.read_csv(descargar("aves.csv"))
ESPECIES = sorted(aves.categoria)
NOMBRE = dict(zip(aves.categoria, aves.nombre))
prueba = pd.read_csv(descargar("aves_prueba.csv"))
fotos_prueba = abrir_zip("aves_prueba.zip")
fotos_sin_etiqueta = abrir_zip("aves_sin_etiqueta.zip")
por_archivo = dict(zip(prueba.archivo, prueba.id))
ids_prueba = [por_archivo[os.path.relpath(f, "archivos_curso").replace(os.sep, "/")] for f in fotos_prueba]
_cache = {}

def vectores_prueba(modelo=None):
    nombre = modelo or MODELO
    if ("prueba", nombre) not in _cache:
        inicio = time.time()
        _cache[("prueba", nombre)] = vectores_de_fotos(fotos_prueba, nombre)
        print(f"{miles(len(fotos_prueba))} fotos de prueba en vectores con {nombre} en {time.time() - inicio:.0f} s.")
    return _cache[("prueba", nombre)]

def vectores_sin_etiqueta(modelo=None):
    nombre = modelo or MODELO
    if ("sin", nombre) not in _cache:
        _cache[("sin", nombre)] = vectores_de_fotos(fotos_sin_etiqueta, nombre)
    return _cache[("sin", nombre)]

print(f"Listo: {miles(len(fotos_prueba))} fotos de prueba, {len(fotos_sin_etiqueta)} sin etiqueta, {len(ESPECIES)} especies.",
      "Con GPU." if DISPOSITIVO == "cuda" else "Sin GPU: convertir las fotos tarda unos minutos, una sola vez.")

In [ ]:
# @title Funciones para describir, combinar y validar (no hace falta cambiarlas)
def _texto_de(c, descripciones):
    d = descripciones[c]
    return d if isinstance(d, (list, tuple)) else [d]

def vectores_de_descripciones(descripciones, modelo=None):
    """Un vector por especie: el promedio de los vectores de sus frases (puede ser una sola)."""
    T = np.stack([vectores_de_textos(_texto_de(c, descripciones), modelo).mean(0) for c in ESPECIES])
    return T / np.linalg.norm(T, axis=1, keepdims=True)

def _estandarizar(S):
    return (S - S.mean(1, keepdims=True)) / S.std(1, keepdims=True)

def puntajes(X, descripciones, mis_etiquetas=None, peso=0.0, modelo=None, sin_etiqueta=None):
    """Un puntaje por foto y especie: el parecido con la descripción y, si hay fotos etiquetadas por ti, el parecido
    con el promedio de esas fotos (multiplicado por `peso`)."""
    S = _estandarizar(X @ vectores_de_descripciones(descripciones, modelo).T)
    if mis_etiquetas and peso > 0:
        P = sin_etiqueta if sin_etiqueta is not None else vectores_sin_etiqueta(modelo)
        protos = []
        for c in ESPECIES:
            mias = [i for i, e in mis_etiquetas.items() if e == c]
            protos.append(P[mias].mean(0) if mias else np.zeros(P.shape[1]))
        protos = np.stack(protos)
        protos /= np.maximum(np.linalg.norm(protos, axis=1, keepdims=True), 1e-9)
        S = S + peso * _estandarizar(X @ protos.T)
    return S

def predecir(X, descripciones, mis_etiquetas=None, peso=0.0, modelo=None):
    return np.array(ESPECIES)[puntajes(X, descripciones, mis_etiquetas, peso, modelo).argmax(1)]

def validar(descripciones, mis_etiquetas, peso=0.0, modelo=None):
    """Exactitud con TUS fotos etiquetadas. Cada foto se predice sin usarla a ella misma como ejemplo."""
    if not mis_etiquetas:
        print("Etiqueta algunas fotos en MIS_ETIQUETAS para poder validar.")
        return None
    P = vectores_sin_etiqueta(modelo)
    aciertos = []
    for i, real in mis_etiquetas.items():
        otras = {j: e for j, e in mis_etiquetas.items() if j != i}
        p = np.array(ESPECIES)[puntajes(P[[i]], descripciones, otras, peso, modelo, P).argmax(1)][0]
        aciertos.append(p == real)
    return float(np.mean(aciertos))

In [ ]:
# @title Función para enviar tus predicciones (no hace falta cambiarla)
import getpass, json, urllib.error

PLATAFORMA = os.environ.get("ESTUDIA_PLATAFORMA", "https://estud-ia.eslopezag.com")
_CLAVE = os.environ.get("ESTUDIA_CLAVE", "")

def enviar(ids, predicciones, nota=""):
    """Envía tus predicciones a la plataforma del curso y muestra tu exactitud en la tabla.

    ids: los ids de las fotos de prueba. predicciones: una especie por foto, como "mirla".
    nota: un texto corto para recordar qué probaste (opcional)."""
    global _CLAVE
    if not _CLAVE:
        _CLAVE = getpass.getpass("Pega tu clave (plataforma › Competencia › Copiar) y oprime Enter: ").strip()
    cuerpo = json.dumps({"ids": [int(i) for i in ids], "predicciones": [str(p) for p in predicciones],
                         "nota": str(nota)[:80]}).encode()
    req = urllib.request.Request(f"{PLATAFORMA}/api/competencia/s08/envios", data=cuerpo, method="POST",
                                 headers={"Content-Type": "application/json", "Authorization": f"Bearer {_CLAVE}",
                                          "User-Agent": "EstudIA-notebook/1.0"})
    try:
        with urllib.request.urlopen(req, timeout=60) as r:
            res = json.loads(r.read())
    except urllib.error.HTTPError as e:
        res = json.loads(e.read() or b"{}")
        if e.code == 401:
            _CLAVE = ""  # la próxima vez la pide de nuevo
        print("No se pudo enviar:", res.get("error", f"error {e.code}"))
        return None
    except urllib.error.URLError:
        print("No hay conexión con la plataforma. Revisa tu internet o pregúntale al profesor si está encendida.")
        return None
    metrica = res.get("metrica", "Exactitud")
    print(f"Envío {res['envio']} recibido. {metrica} en la tabla: {pct(res.get('puntaje', res['exactitud']))}.")
    print(f"Tu mejor resultado: {pct(res['mejor'])} · puesto {res['puesto']} de {res['participantes']} ({res['alias']}).")
    return res

---
## 1. Las aves

Algunas fotos sin etiqueta, para que veas cómo son las fotos (de lejos, de cerca, en cables, en ramas…).

In [ ]:
print(aves.to_string(index=False))
mostrar(fotos_sin_etiqueta[:12], [f"sin etiqueta {i}" for i in range(12)], columnas=6)

---
## 2. Tu primer envío: los nombres

Cada especie con una frase. Así como vienen, las frases solo dicen el nombre de cada ave. Funcionan, pero dejan mucho
por ganar. En cada especie puedes poner una frase o una lista de frases (el notebook promedia sus vectores).

In [ ]:
DESCRIPCIONES = {c: f"una foto de un {NOMBRE[c]}" for c in ESPECIES}
DESCRIPCIONES

In [ ]:
X = vectores_prueba()
predicciones = predecir(X, DESCRIPCIONES)
print("Fotos que el modelo le da a cada especie:", pd.Series(predicciones).value_counts().to_dict())
respuesta = enviar(ids_prueba, predicciones, nota="nombres en español")
registrar(modelo=MODELO, descripciones="nombres en español", mis_fotos=0, peso=0,
          validacion=None, tabla=round(100 * respuesta["puntaje"], 1) if respuesta else None)

---
## 3. Mejora tus descripciones

Escribe aquí tus descripciones y vuelve a ejecutar. Si etiquetaste fotos (sección 4), `validar` te dice qué tan bien
le va a tu versión con esas fotos, sin gastar envíos. Con pocas fotos es una medida aproximada, y algo pesimista: cada
foto se predice sin usarla como ejemplo, así que su especie queda con una foto menos. Sirve para comparar dos
versiones entre sí.

In [ ]:
MIS_DESCRIPCIONES = dict(DESCRIPCIONES)        # cópialas y cámbialas, por ejemplo:
# MIS_DESCRIPCIONES["azulejo"] = "una foto de un pájaro pequeño de color azul grisáceo claro"
MIS_ETIQUETAS = {}                             # sección 4: {número de foto sin etiqueta: especie}
PESO_FOTOS = 0.0                               # sección 4: cuánto pesan tus fotos etiquetadas

v = validar(MIS_DESCRIPCIONES, MIS_ETIQUETAS, PESO_FOTOS)
if v is not None:
    print(f"Validación con tus {len(MIS_ETIQUETAS)} fotos etiquetadas: {pct(v)}")

In [ ]:
if MIS_DESCRIPCIONES == DESCRIPCIONES and not (MIS_ETIQUETAS and PESO_FOTOS):
    print("Todavía no cambiaste nada: escribe tus descripciones (o etiqueta fotos) arriba y vuelve a ejecutar.")
else:
    predicciones = predecir(X, MIS_DESCRIPCIONES, MIS_ETIQUETAS, PESO_FOTOS)
    nota = "mis descripciones" + (f" + {len(MIS_ETIQUETAS)} fotos, peso {PESO_FOTOS}" if MIS_ETIQUETAS and PESO_FOTOS
                                  else "")
    respuesta = enviar(ids_prueba, predicciones, nota=nota)
    registrar(modelo=MODELO, descripciones=nota, mis_fotos=len(MIS_ETIQUETAS), peso=PESO_FOTOS,
              validacion=None if v is None else round(100 * v, 1),
              tabla=round(100 * respuesta["puntaje"], 1) if respuesta else None)
ver_registro()

---
## 4. Etiqueta fotos tú mismo

Mira las fotos sin etiqueta y anota en `MIS_ETIQUETAS` las que reconozcas: `{número: especie}`. El promedio de los
vectores de tus fotos de cada especie funciona como una descripción más, hecha con fotos en vez de palabras. Súbele
`PESO_FOTOS` (por ejemplo, 0,5) para combinarla con tus descripciones. Tus fotos etiquetadas también te sirven para
validar.

In [ ]:
DESDE = 0          # cambia este número para ver otras fotos (hay 400: de la 0 a la 399)
mostrar(fotos_sin_etiqueta[DESDE:DESDE + 24], [str(i) for i in range(DESDE, DESDE + 24)], columnas=8, tamano=1.6)

---
## 5. Pistas para subir en la tabla

Prueba una cosa a la vez y mira tu validación y tu registro.

<details><summary>Pista 1</summary>

Mira las fotos: ¿qué distingue a cada ave? El modelo no sabe qué es un “azulejo” (en España, un azulejo es una
baldosa). Describe cómo se ve: “una foto de un pájaro pequeño de color azul grisáceo claro”. Colores, tamaño, forma del
pico o de la cola, dónde suele estar.
</details>

<details><summary>Pista 2</summary>

Varias frases por especie, en una lista: el notebook promedia sus vectores. Por ejemplo, el nombre y dos descripciones
distintas.
</details>

<details><summary>Pista 3</summary>

Prueba tus descripciones **en inglés**. Este modelo entiende español, pero aprendió sobre todo con pies de foto en
inglés, que son la mayoría en internet: hay muchas más fotos de aves descritas en inglés. Ojo: en inglés, las
descripciones ayudan mucho más que los nombres comunes de las aves. Y puedes mezclar frases en español y en inglés.
</details>

<details><summary>Pista 4</summary>

Etiqueta fotos tú mismo (sección 4): con tres o cuatro fotos por especie ya se nota. Empieza con `PESO_FOTOS = 0.5`
y compara con `validar`. Mientras más fotos etiquetes, mejor validas.
</details>

<details><summary>Pista 5</summary>

¿Qué especies confunde tu modelo? El sirirí y el bichofué tienen el pecho amarillo; el copetón, la mirla y la torcaza
son pardos. Fíjate en lo que los distingue y ponlo en las descripciones.
</details>

<details><summary>Pista 6</summary>

Otro modelo: `MODELO = "openai/clip-vit-base-patch32"` (CLIP original, solo inglés) conoce los nombres comunes de las
aves **en inglés**, como “Blue-gray Tanager”. ¿Por qué crees que con los nombres en español le va tan mal?
</details>

**Una advertencia.** Si eliges tus cambios mirando solo la tabla en vivo, terminas ajustando tus descripciones a esa
mitad de la prueba. El resultado final se calcula con la otra mitad. Valida con tus fotos etiquetadas.

**Fuente de las fotos:** observaciones de iNaturalist hechas en Colombia, con grado de investigación y licencia CC0,
CC BY o CC BY-SA. El autor y la licencia de cada foto están en `creditos_fotos.csv`. Las fotos de prueba y las sin
etiqueta son de personas distintas.